# INPUTS

In [9]:
# REQUIRED inputs you set: # fn1
pl_name = "https://soundcloud.com/user862007976/sets/salsa?si=13f9da5ee0074af1aa9b6e7b919f6e94&utm_source=clipboard&utm_medium=text&utm_campaign=social_sharing"
# REQUIRED inputs you set: fn2
OUT_DIR = "/Users/yerik/Downloads/_soundcloud_audio"  # your target folder (will be created)
EXT     = "mp3"                                       # "mp3" (320), "wav", or "aiff"
KBPS    = 320                                         # only for mp3
COOKIES = None                                        # e.g., "/Users/yerik/Downloads/cookies.txt" if needed

# RUN

In [11]:
# Get urls from playlists 
df_tracks = _soundcloud_1808_playlist_GET_df_urls(pl_name)
df_tracks

TQM • Scraping playlists: 100%|███████████████████████████████████████████| 1/1 [00:01<00:00,  1.23s/playlist]


,src_playlist,track_title,track_artist,track_url,track_id,error
0,https://soundcloud.com/user862007976/sets/sals...,TITO NIEVES-SeÃ±ora Ley ''Intro En Vivo'' CALL...,jvdeejay,https://soundcloud.com/jvdeejay/tito-nieves-se...,76080354,
1,https://soundcloud.com/user862007976/sets/sals...,Hacha y Machete Hector Lavoe,chulaconsabor4u,https://soundcloud.com/chulaconsabor4u/hacha-y...,21676899,
2,https://soundcloud.com/user862007976/sets/sals...,,,,84039119,
3,https://soundcloud.com/user862007976/sets/sals...,,,,55756740,
4,https://soundcloud.com/user862007976/sets/sals...,,,,16490112,
5,https://soundcloud.com/user862007976/sets/sals...,,,,83888491,
6,https://soundcloud.com/user862007976/sets/sals...,,,,187568615,
7,https://soundcloud.com/user862007976/sets/sals...,,,,5161824,
8,https://soundcloud.com/user862007976/sets/sals...,,,,29900346,
9,https://soundcloud.com/user862007976/sets/sals...,,,,31350378,


In [10]:
# Get urls from playlists 
df_tracks = _soundcloud_1808_playlist_GET_df_urls(pl_name)
#df_tracks

# REQUIRED inputs you set:
OUT_DIR = "/Users/yerik/Downloads/_soundcloud_audio"  # your target folder (will be created)
EXT     = "mp3"                                       # "mp3" (320), "wav", or "aiff"
KBPS    = 320                                         # only for mp3
COOKIES = None                                        # e.g., "/Users/yerik/Downloads/cookies.txt" if needed

# df_tracks must exist and contain 'src_playlist' with SoundCloud URLs
df_tracks = _sc_1808_df_GET_audio_for_src_playlist_APPEND(
    df_tracks,
    out_dir=OUT_DIR,
    ext=EXT,
    prefer_bitrate=KBPS,
    cookies_path=COOKIES
)

# (Optional) Save the augmented DF
# df_tracks.to_csv("/Users/yerik/Downloads/_soundcloud_audio/_dl_log.csv", index=False)


TQM • Downloading SoundCloud audio:   0%|                                             | 0/24 [00:00<?, ?url/s]

TQM • Downloading SoundCloud audio: 100%|████████████████████████████████████| 24/24 [01:51<00:00,  4.63s/url]


# FUNCTIONS 

#### # Get urls from playlists 

In [2]:
# # -----######-----###### CORE IMPORTABLE FUNCTION (SoundCloud Playlist Scraper) -----######-----###### #
# import re, json, time
# import requests
# import pandas as pd
# from tqdm import tqdm

# def _soundcloud_1808_playlist_GET_df_urls(playlist_links):
#     def _as_list(x):
#         if isinstance(x, (list, tuple, pd.Series, pd.Index)): return list(x)
#         return [x]

#     def _request(url):
#         return requests.get(url, headers={
#             "User-Agent": "Mozilla/5.0",
#             "Accept": "text/html"
#         }, timeout=20)

#     def _scrape_tracks(pl_url):
#         html = _request(pl_url).text
#         m = re.search(r"window\.__sc_hydration\s*=\s*(\[[\s\S]*?\])\s*;", html)
#         if not m: return []
#         try: hydration = json.loads(m.group(1))
#         except: return []
#         pl_objs = [o for o in hydration if isinstance(o, dict) and o.get("hydratable","").startswith("playlist")]
#         tracks = []
#         for obj in pl_objs:
#             data = obj.get("data") or {}
#             for t in data.get("tracks") or []:
#                 user = t.get("user") or {}
#                 tracks.append({
#                     "src_playlist": pl_url,
#                     "track_title": t.get("title",""),
#                     "track_artist": user.get("username",""),
#                     "track_url": t.get("permalink_url",""),
#                     "track_id": t.get("id"),
#                     "error": ""
#                 })
#         return tracks

#     links = _as_list(playlist_links)
#     rows = []
#     for pl in tqdm(links, desc="TQM • Scraping playlists", unit="playlist"):
#         try: rows.extend(_scrape_tracks(pl) or [{"src_playlist":pl,"track_title":"","track_artist":"","track_url":"","track_id":None,"error":"no_tracks"}])
#         except Exception as e: rows.append({"src_playlist":pl,"track_title":"","track_artist":"","track_url":"","track_id":None,"error":f"{e}"})
#         time.sleep(0.8)
#     return pd.DataFrame(rows, columns=["src_playlist","track_title","track_artist","track_url","track_id","error"])
# -----######-----###### CORE IMPORTABLE FUNCTION (SC Track Table → Enriched + new_name) -----######-----###### #
import re
import pandas as pd
from datetime import datetime
from tqdm import tqdm

def _soundcloud_1808_enrich_GET_df_named(df_in, genre="", purchase_date=""):
    """
    Input:
      - df_in: pandas DataFrame with columns at least ['track_title','track_artist','track_url'] (others ok)
      - genre: optional string; if empty, function will prompt once via input()
      - purchase_date: optional 'YYYY-MM-DD'; empty keeps blank purchase fields
    Output:
      - DataFrame with appended columns:
        ['genre','mix_name','remixers','label','key','bpm',
         'release_year','release_month','release_day',
         'purchase_year','purchase_month','purchase_day',
         'new_name']
    Notes:
      - We do NOT invent values: only parse what’s present in titles/strings or params.
      - Parsing heuristics: title → mix/remixers, key (music/Camelot), bpm hints like '128 BPM' or '128bpm'.
    """
    df = df_in.copy()

    if not isinstance(df, pd.DataFrame):
        raise ValueError("df_in must be a pandas DataFrame")

    for col in ["track_title","track_artist","track_url"]:
        if col not in df.columns:
            df[col] = ""

    if not genre:
        try:
            genre = input("Enter GENRE for this batch: ").strip()
        except Exception:
            genre = ""
    df["genre"] = genre

    # Purchase date split (optional, no defaults)
    py = pm = pdm = ""
    if purchase_date:
        try:
            pdt = datetime.strptime(purchase_date, "%Y-%m-%d")
            py, pm, pdm = str(pdt.year), f"{pdt.month:02d}", f"{pdt.day:02d}"
        except Exception:
            py = pm = pdm = ""
    df["purchase_year"] = py
    df["purchase_month"] = pm
    df["purchase_day"] = pdm

    # Prepare empty columns
    add_cols = ["mix_name","remixers","label","key","bpm","release_year","release_month","release_day"]
    for c in add_cols:
        if c not in df.columns:
            df[c] = ""

    # Heuristic regex sets
    rx_mix = re.compile(r"\(([^)]*?(?:mix|edit|version|dub|instrumental)[^)]*)\)", re.IGNORECASE)
    rx_remix = re.compile(r"\(([^)]*?remix[^)]*)\)", re.IGNORECASE)
    rx_key_music = re.compile(r"\b([A-G](?:#|b)?\s?(?:maj(?:or)?|min(?:or)?|m|M))\b")
    rx_key_camelot = re.compile(r"\b(1[0-2]|[1-9])[AB]\b", re.IGNORECASE)
    rx_bpm = re.compile(r"\b(\d{2,3})\s?bpm\b", re.IGNORECASE)
    rx_bpm_brackets = re.compile(r"\[(\d{2,3})\]")  # sometimes titles have [128]
    rx_feat = re.compile(r"\b(feat\.?|ft\.?)\b", re.IGNORECASE)

    titles = df["track_title"].fillna("").astype(str).tolist()
    artists = df["track_artist"].fillna("").astype(str).tolist()

    mix_list, remixers_list, key_list, bpm_list = [], [], [], []

    for t in tqdm(titles, desc="TQM • Parsing titles", unit="track"):
        # Mix name
        mix = ""
        m_mix = rx_mix.search(t)
        if m_mix:
            mix = m_mix.group(1).strip()

        # Remixers (if “… Remix” found in parentheses, take before 'Remix')
        rem = ""
        m_rem = rx_remix.search(t)
        if m_rem:
            inner = m_rem.group(1)
            parts = re.split(r"remix", inner, flags=re.IGNORECASE)
            rem = parts[0].strip(" -&x,").strip()

        # Key
        key_val = ""
        m_km = rx_key_music.search(t)
        if m_km:
            key_val = m_km.group(1).strip().replace("major","maj").replace("minor","min")
        else:
            m_kc = rx_key_camelot.search(t)
            if m_kc:
                key_val = m_kc.group(0).upper()

        # BPM
        bpm_val = ""
        m_bpm = rx_bpm.search(t)
        if m_bpm:
            bpm_val = m_bpm.group(1)
        else:
            m_b2 = rx_bpm_brackets.search(t)
            if m_b2:
                bpm_val = m_b2.group(1)

        mix_list.append(mix)
        remixers_list.append(rem)
        key_list.append(key_val)
        bpm_list.append(bpm_val)

    df["mix_name"] = df["mix_name"].where(df["mix_name"].ne(""), mix_list)
    df["remixers"] = df["remixers"].where(df["remixers"].ne(""), remixers_list)
    df["key"] = df["key"].where(df["key"].ne(""), key_list)
    df["bpm"] = df["bpm"].where(df["bpm"].ne(""), bpm_list)

    # Basic cleanup for artist field if title carries “feat/ft”
    clean_artists = []
    for a, t in zip(artists, titles):
        if rx_feat.search(t):
            clean_artists.append(a)  # do not append featured for now (no assumptions)
        else:
            clean_artists.append(a)
    df["track_artist"] = clean_artists

    # We do not guess label or release date; keep any existing if present
    def nz(x): return "" if pd.isna(x) else str(x)

    df["new_name"] = (
        "TRkw_" + df["track_title"].map(nz) +
        "_ARkw_" + df["track_artist"].map(nz) +
        "_MXkw_" + df["mix_name"].map(nz) +
        "_KYkw_" + df["key"].map(nz) +
        "_BPkw_" + df["bpm"].map(nz) +
        "_GNkw_" + df["genre"].map(nz) +
        "_RMkw_" + df["remixers"].map(nz) +
        "_LBkw_" + df["label"].map(nz) +
        "_RYkw_" + df["release_year"].map(nz) + "_" + df["release_month"].map(nz) + "_" + df["release_day"].map(nz) +
        "_PYkw_" + df["purchase_year"].map(nz) + "_" + df["purchase_month"].map(nz) + "_" + df["purchase_day"].map(nz)
    ).str.replace(r"[\/\\:*?\"<>|]", "_", regex=True)

    return df


##### 2

In [4]:
# -----######-----###### CORE IMPORTABLE FUNCTION (SoundCloud DF Downloader — DF['src_playlist'] → files) -----######-----###### #
import os, re, json, shutil
from pathlib import Path
from datetime import datetime
from tqdm import tqdm

try:
    import pandas as pd
except Exception:
    pd = None


# --- internal: env checks (no ASCII) ---
def _sc__ensure_ffmpeg():
    from shutil import which
    return which("ffmpeg") is not None

def _sc__safe_name(s):
    # filesystem-safe slug
    s = str(s or "").strip()
    s = re.sub(r"[\\/:*?\"<>|\n\r\t]", "_", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s[:200] if s else "unnamed"

def _sc__build_opts(out_dir, ext, prefer_bitrate, archive_path, cookies_path):
    # postproc: convert to target ext
    postproc = []
    if ext.lower() in ("mp3", "wav", "aiff"):
        postproc = [{
            "key": "FFmpegExtractAudio",
            "preferredcodec": ext.lower(),
            "preferredquality": str(prefer_bitrate) if ext.lower()=="mp3" else "0",
        }]

    # Filename template: Uploader - Title [id].ext inside out_dir
    outtmpl = str(Path(out_dir) / "%(uploader)s - %(title)s [%(id)s].%(ext)s")

    opts = {
        "outtmpl": outtmpl,
        "noplaylist": False,              # if a playlist URL appears, we’ll let yt-dlp handle entries
        "quiet": True,
        "no_warnings": True,
        "ignoreerrors": False,
        "retries": 5,
        "continuedl": True,
        "format": "bestaudio/best",
        "concurrent_fragment_downloads": 4,
        "writethumbnail": False,
        "addmetadata": True,
        "prefer_ffmpeg": True,
        "postprocessors": postproc,
        "restrictfilenames": False,
        "windowsfilenames": False,
        "nooverwrites": True,            # do NOT overwrite existing files
        "download_archive": str(archive_path),  # skip already downloaded ids
    }
    if cookies_path:
        opts["cookiefile"] = str(cookies_path)
    return opts

def _sc__extract_saved_path(info):
    cand = None
    if not info:
        return None
    # Try direct
    if "requested_downloads" in info and info["requested_downloads"]:
        cand = info["requested_downloads"][0].get("filepath")
    # Playlist entry?
    if not cand and "entries" in info and info["entries"]:
        e = info["entries"][0]
        if e and "requested_downloads" in e and e["requested_downloads"]:
            cand = e["requested_downloads"][0].get("filepath")
        elif e and "filepath" in e:
            cand = e["filepath"]
    if not cand and "filepath" in info:
        cand = info["filepath"]
    return Path(cand) if cand else None

def _sc__download_one(url, out_dir, ext, prefer_bitrate, archive_path, cookies_path):
    try:
        import yt_dlp
    except Exception as e:
        raise RuntimeError("yt-dlp not installed. Run: pip install yt-dlp") from e

    if ext.lower() in ("mp3","wav","aiff") and not _sc__ensure_ffmpeg():
        raise RuntimeError("ffmpeg not found. Install with: brew install ffmpeg")

    ydl_opts = _sc__build_opts(out_dir, ext, prefer_bitrate, archive_path, cookies_path)

    # progress hook (quiet by default; reserved for future per-fragment logs)
    def _hook(_d): 
        pass
    ydl_opts["progress_hooks"] = [_hook]

    meta = {
        "source_url": url, "title": None, "uploader": None, "duration_sec": None,
        "id": None, "ext": None, "requested_ext": ext.lower(), "filepath": None,
        "filesize_approx": None, "filesize_bytes": None, "error": None, "status": None
    }

    # Detect “already downloaded” quickly via archive line (best-effort);
    # yt-dlp itself will skip by archive and return fast.
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(url, download=True)
            # When skipped by archive, info may be None → try to probe id via “simulate”
            if not info:
                try:
                    sim_opts = ydl_opts.copy()
                    sim_opts.update({"skip_download": True})
                    with yt_dlp.YoutubeDL(sim_opts) as ydl_sim:
                        info = ydl_sim.extract_info(url, download=False)
                        meta["status"] = "skipped_archive"
                except Exception:
                    meta["status"] = "skipped_archive"
            else:
                meta["status"] = "downloaded"

        if info:
            meta["title"] = info.get("title")
            meta["uploader"] = info.get("uploader")
            meta["duration_sec"] = info.get("duration")
            meta["id"] = info.get("id")
            meta["ext"] = info.get("ext")
            meta["filesize_approx"] = info.get("filesize_approx")

        saved = _sc__extract_saved_path(info)
        if saved and saved.exists():
            meta["filepath"] = str(saved)
            meta["filesize_bytes"] = saved.stat().st_size
        else:
            # If archive skip, try to resolve the existing path on disk by building the expected pattern:
            if meta.get("id") and meta.get("title") and meta.get("uploader"):
                # Try any ext since postproc could differ
                base_glob = f"{_sc__safe_name(meta['uploader'])} - {_sc__safe_name(meta['title'])} [{meta['id']}]"
                candidates = list(Path(out_dir).glob(base_glob + ".*"))
                if candidates:
                    meta["filepath"] = str(candidates[0])
                    meta["filesize_bytes"] = candidates[0].stat().st_size
        return meta

    except yt_dlp.utils.DownloadError as e:
        meta["error"] = f"DownloadError: {e}"
        meta["status"] = "failed"
        return meta
    except Exception as e:
        meta["error"] = str(e)
        meta["status"] = "failed"
        return meta


# -----######-----###### MAIN IMPORTABLE FUNCTION -----######-----###### #
def _sc_1808_df_GET_audio_for_src_playlist_APPEND(
    df_tracks,
    out_dir,                  # REQUIRED: you provide this absolute path
    ext="mp3",                # "mp3" (320), "wav", "aiff"
    prefer_bitrate=320,       # only for mp3
    cookies_path=None         # optional path to cookies.txt for authenticated access
):
    """
    From df_tracks['src_playlist'] SoundCloud URLs, download audio into out_dir.
    - Never overwrites existing files (yt-dlp nooverwrites + archive).
    - Re-running will only add new tracks (download archive).
    - Appends result columns to df_tracks and returns the augmented DF.

    Columns appended/updated:
      dl_title, dl_uploader, dl_duration_sec, dl_id, dl_ext, dl_requested_ext,
      dl_filepath, dl_filesize_approx, dl_filesize_bytes, dl_error, dl_status

    Notes:
      * For private/unlisted but accessible to you, pass a cookies.txt exported from your browser.
      * Only download content you own or have rights to and respect the site’s TOS.
    """
    if pd is None:
        raise RuntimeError("pandas is required. Run: pip install pandas")

    if "src_playlist" not in df_tracks.columns:
        raise ValueError("DataFrame must contain a 'src_playlist' column with SoundCloud URLs.")

    out_dir = Path(out_dir).expanduser().resolve()
    out_dir.mkdir(parents=True, exist_ok=True)

    archive_path = out_dir / "_download_archive.txt"   # tracks already fetched
    archive_path.touch(exist_ok=True)

    # Prepare output columns
    add_cols = [
        "dl_title","dl_uploader","dl_duration_sec","dl_id","dl_ext","dl_requested_ext",
        "dl_filepath","dl_filesize_approx","dl_filesize_bytes","dl_error","dl_status"
    ]
    df_out = df_tracks.copy()
    for c in add_cols:
        if c not in df_out.columns:
            df_out[c] = None

    urls = df_out["src_playlist"].astype(str).fillna("").tolist()

    # TQM BAR
    for i, url in enumerate(tqdm(urls, desc="TQM • Downloading SoundCloud audio", unit="url")):
        url_s = url.strip()
        if not url_s:
            df_out.at[i, "dl_status"] = "skip_empty"
            continue
        md = _sc__download_one(
            url=url_s,
            out_dir=out_dir,
            ext=ext,
            prefer_bitrate=prefer_bitrate,
            archive_path=archive_path,
            cookies_path=(Path(cookies_path).expanduser() if cookies_path else None)
        )

        df_out.at[i, "dl_title"]            = md.get("title")
        df_out.at[i, "dl_uploader"]         = md.get("uploader")
        df_out.at[i, "dl_duration_sec"]     = md.get("duration_sec")
        df_out.at[i, "dl_id"]               = md.get("id")
        df_out.at[i, "dl_ext"]              = md.get("ext")
        df_out.at[i, "dl_requested_ext"]    = md.get("requested_ext")
        df_out.at[i, "dl_filepath"]         = md.get("filepath")
        df_out.at[i, "dl_filesize_approx"]  = md.get("filesize_approx")
        df_out.at[i, "dl_filesize_bytes"]   = md.get("filesize_bytes")
        df_out.at[i, "dl_error"]            = md.get("error")
        df_out.at[i, "dl_status"]           = md.get("status")

    return df_out


# better function try 

In [ ]:
# # -----######-----###### CORE IMPORTABLE FUNCTION (SoundCloud DF Downloader — Append + Rate Limit + Backoff + Log) -----######-----###### #
# import os, re, time, random, math
# from pathlib import Path
# from datetime import datetime
# from tqdm import tqdm
# import pandas as pd

# def _sc__ensure_ffmpeg():
#     from shutil import which
#     return which("ffmpeg") is not None

# def _sc__build_opts(out_dir, ext, prefer_bitrate, archive_path, cookies_path, max_concurrent_frags=2):
#     postproc = []
#     if ext.lower() in ("mp3", "wav", "aiff"):
#         postproc = [{
#             "key": "FFmpegExtractAudio",
#             "preferredcodec": ext.lower(),
#             "preferredquality": str(prefer_bitrate) if ext.lower()=="mp3" else "0",
#         }]
#     outtmpl = str(Path(out_dir) / "%(uploader)s - %(title)s.%(ext)s")  # clean names

#     opts = {
#         "outtmpl": outtmpl,
#         "noplaylist": False,
#         "quiet": True,
#         "no_warnings": True,
#         "ignoreerrors": False,
#         "retries": 5,
#         "continuedl": True,
#         "format": "bestaudio/best",
#         "concurrent_fragment_downloads": int(max(1, max_concurrent_frags)),
#         "addmetadata": True,
#         "prefer_ffmpeg": True,
#         "postprocessors": postproc,
#         "nooverwrites": True,                     # never overwrite on disk
#         "download_archive": str(archive_path),    # skip duplicates by ID
#     }
#     if cookies_path:
#         opts["cookiefile"] = str(Path(cookies_path).expanduser())
#     return opts

# def _sc__extract_saved_path(info):
#     if not info:
#         return None
#     if "requested_downloads" in info and info["requested_downloads"]:
#         p = info["requested_downloads"][0].get("filepath")
#         return Path(p) if p else None
#     if "filepath" in info:
#         return Path(info["filepath"])
#     if "entries" in info and info["entries"]:
#         e = info["entries"][0]
#         if e and "requested_downloads" in e and e["requested_downloads"]:
#             p = e["requested_downloads"][0].get("filepath")
#             return Path(p) if p else None
#         if e and "filepath" in e:
#             return Path(e["filepath"])
#     return None

# def _sc__download_one(url, out_dir, ext, prefer_bitrate, archive_path, cookies_path, max_concurrent_frags):
#     import yt_dlp
#     if ext.lower() in ("mp3","wav","aiff") and not _sc__ensure_ffmpeg():
#         raise RuntimeError("ffmpeg not found. Install with: brew install ffmpeg")

#     ydl_opts = _sc__build_opts(out_dir, ext, prefer_bitrate, archive_path, cookies_path, max_concurrent_frags)

#     res = {
#         "timestamp": datetime.now().isoformat(timespec="seconds"),
#         "src_playlist": url, "dl_title": None, "dl_uploader": None, "dl_duration_sec": None,
#         "dl_id": None, "dl_ext": None, "dl_requested_ext": ext.lower(),
#         "dl_filepath": None, "dl_filesize_bytes": None, "dl_error": None, "dl_status": None
#     }

#     try:
#         with yt_dlp.YoutubeDL(ydl_opts) as ydl:
#             info = ydl.extract_info(url, download=True)
#             if not info:
#                 res["dl_status"] = "skipped"   # likely skipped by archive
#                 return res

#         res["dl_title"]        = info.get("title")
#         res["dl_uploader"]     = info.get("uploader")
#         res["dl_duration_sec"] = info.get("duration")
#         res["dl_id"]           = info.get("id")
#         res["dl_ext"]          = info.get("ext")
#         res["dl_status"]       = "downloaded"

#         saved = _sc__extract_saved_path(info)
#         if saved and saved.exists():
#             res["dl_filepath"]       = str(saved)
#             res["dl_filesize_bytes"] = saved.stat().st_size

#     except Exception as e:
#         res["dl_error"]  = str(e)
#         res["dl_status"] = "failed"

#     return res


# # -----######-----###### MAIN IMPORTABLE FUNCTION -----######-----###### #
# def _sc_1808_df_GET_audio_APPEND_safe(
#     df_tracks,
#     out_dir,                      # REQUIRED: absolute path you provide
#     ext="mp3",
#     prefer_bitrate=320,
#     cookies_path=None,
#     existing_log_csv=None,        # optional: path to persistent CSV log (will be created/appended)
#     max_per_minute=30,            # polite rate: max URLs/min (≈ 2 req/s)
#     daily_cap=None,               # optional hard cap of downloads per run/day
#     max_retries_each=3,           # exponential backoff tries per URL
#     max_concurrent_frags=2        # keep per-file fragment concurrency modest
# ):
#     """
#     Append-only SoundCloud downloader with rate limiting, backoff, and persistent CSV logging.
#     Input:
#       - df_tracks: must contain 'src_playlist' column with SoundCloud URLs.
#       - out_dir: folder where audio will be saved (never overwritten).
#     Behavior:
#       - Uses a download archive in out_dir to skip already downloaded IDs.
#       - Clean filenames (Artist - Title.ext).
#       - Appends rows to an existing CSV log if provided.
#       - Respects max_per_minute with random jitter; exponential backoff on failures.
#     Returns:
#       - df_log: a DataFrame of ONLY the rows from this run (you can read/concat the CSV for full history).
#     """
#     if "src_playlist" not in df_tracks.columns:
#         raise ValueError("df_tracks must contain 'src_playlist' with SoundCloud URLs.")

#     out_dir = Path(out_dir).expanduser().resolve()
#     out_dir.mkdir(parents=True, exist_ok=True)
#     archive_path = out_dir / "_download_archive.txt"
#     archive_path.touch(exist_ok=True)

#     # Load existing CSV log (for your reference). We still return just-this-run rows.
#     if existing_log_csv:
#         existing_log_csv = str(Path(existing_log_csv).expanduser())
#         if not Path(existing_log_csv).exists():
#             Path(existing_log_csv).parent.mkdir(parents=True, exist_ok=True)

#     urls = [u.strip() for u in df_tracks["src_playlist"].astype(str).fillna("")]
#     urls = [u for u in urls if u]  # drop blanks

#     # Rate limiting params
#     min_interval = 60.0 / max(1, max_per_minute)  # seconds per URL
#     last_ts = 0.0
#     successes_this_run = 0
#     emitted = []

#     for idx, url in enumerate(tqdm(urls, desc="TQM • Downloading SoundCloud audio", unit="url")):
#         # Daily cap stop
#         if daily_cap is not None and successes_this_run >= daily_cap:
#             emitted.append({
#                 "timestamp": datetime.now().isoformat(timespec="seconds"),
#                 "src_playlist": url,
#                 "dl_title": None, "dl_uploader": None, "dl_duration_sec": None,
#                 "dl_id": None, "dl_ext": None, "dl_requested_ext": ext.lower(),
#                 "dl_filepath": None, "dl_filesize_bytes": None,
#                 "dl_error": "daily_cap_reached", "dl_status": "skipped_cap"
#             })
#             continue

#         # Polite pacing with jitter
#         now = time.time()
#         wait_needed = (last_ts + min_interval) - now
#         if wait_needed > 0:
#             time.sleep(wait_needed)
#         # Add small random jitter to avoid looking robotic
#         time.sleep(random.uniform(0.05, 0.35))
#         last_ts = time.time()

#         # Retry with exponential backoff
#         attempt = 0
#         result = None
#         while attempt < max_retries_each:
#             result = _sc__download_one(
#                 url=url,
#                 out_dir=out_dir,
#                 ext=ext,
#                 prefer_bitrate=prefer_bitrate,
#                 archive_path=archive_path,
#                 cookies_path=cookies_path,
#                 max_concurrent_frags=max_concurrent_frags
#             )

#             # Consider "downloaded" or "skipped" by archive a success
#             if result.get("dl_status") in ("downloaded", "skipped"):
#                 break

#             # Backoff on failure (network/429/etc.)
#             attempt += 1
#             if attempt < max_retries_each:
#                 sleep_s = (2 ** attempt) + random.uniform(0, 0.6)
#                 time.sleep(sleep_s)

#         emitted.append(result)
#         if result.get("dl_status") in ("downloaded", "skipped"):
#             successes_this_run += 1

#         # Persist row-by-row to CSV to never lose progress
#         if existing_log_csv:
#             pd.DataFrame([result]).to_csv(existing_log_csv, mode="a", header=not Path(existing_log_csv).exists(), index=False)

#     df_run = pd.DataFrame(emitted)
#     return df_run


# # REQUIRED: your output folder
# OUT_DIR = "/Users/yerik/Downloads/_soundcloud_audio"

# # Optional persistent log (grows forever, append-only)
# LOG_CSV = "/Users/yerik/Downloads/_soundcloud_audio/_download_log.csv"

# # mp3/wav/aiff settings
# EXT   = "mp3"
# KBPS  = 320
# COOKS = None  # e.g. "/Users/yerik/Downloads/cookies.txt" if needed

# # Politeness knobs (tune if you see throttling)
# MAX_PER_MIN   = 30     # ≈2 req/s is usually safe; drop to 10 if you see 429s
# DAILY_CAP     = None   # e.g. 300 if you want a hard ceiling
# RETRIES_EACH  = 3
# FRAG_CONCUR   = 2

# df_log_run = _sc_1808_df_GET_audio_APPEND_safe(
#     df_tracks,
#     out_dir=OUT_DIR,
#     ext=EXT,
#     prefer_bitrate=KBPS,
#     cookies_path=COOKS,
#     existing_log_csv=LOG_CSV,
#     max_per_minute=MAX_PER_MIN,
#     daily_cap=DAILY_CAP,
#     max_retries_each=RETRIES_EACH,
#     max_concurrent_frags=FRAG_CONCUR
# )

# # Tip: if you want the full history in memory:
# # import pandas as pd
# # df_full = pd.read_csv(LOG_CSV)
